<a href="https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/label_studio_to_nemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Colab to convert label studio exported JSON into NeMO manifest

The google drive version of this colab (for iterating and editing) [is here](https://colab.research.google.com/drive/1DLNMdsvrUpIbYtsSVBjdq15oFYiZjyXV?authuser=1#scrollTo=72af-Ue2_BwC). Once you have a unit of edit completed, you can check-in the drive version into [this github directory](https://github.com/watch-duty/radio-transcription/tree/main/model/colabs).

In [ ]:
#@title Imports
import json
import os

input_json = "/usr/local/google/home/varungulshan/radio-transcription/model/data/label_studio_exports/playground_export.json"  #@param
output_jsonl = "/usr/local/google/home/varungulshan/radio-transcription/model/data/manifests/playground_export_nemo.json"  #@param
label_studio_data_prefix = "/data/upload/1/"  #@param
actual_data_prefix = "/usr/local/google/home/varungulshan/watchduty-asr/data_exports/playground_export/"  #@param

In [ ]:
#@title Helper function

def convert_to_nemo_manifest(input_file, output_file, prefix_to_remove, prefix_to_add):
  """
  Converts Label Studio JSON export to NVIDIA NeMo JSONL manifest.

  Args:
    input_file (str): Path to the input JSON file.
    output_file (str): Path to the output JSONL file.
    audio_path_prefix (str): Optional prefix to replace/prepend to audio paths
                     if the local paths differ from the upload paths.
  """

  with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

  with open(output_file, 'w', encoding='utf-8') as out:
    # Iterate through each task (file) in the export
    for task in data:
      # Get the base audio file path
      original_audio_path = task.get('data', {}).get('audio')

      # Update data paths prefixs to account for how label studio organizes
      # data.
      if not original_audio_path.startswith(prefix_to_remove):
        raise ValueError(f"Data path doesn't start with prefix {original_audio_path}")

      audio_filepath = os.path.join(prefix_to_add, original_audio_path[len(prefix_to_remove):])

      # Locate annotations
      annotations = task.get('annotations', [])
      if not annotations:
        continue

      # We usually take the most recent annotation (annotations[0] in this format)
      results = annotations[0].get('result', [])

      for item in results:
        # Filter strictly for 'transcription' types
        if item.get('from_name') != 'transcription':
          continue

        value = item.get('value', {})

        # Extract timing and text
        start = value.get('start', 0.0)
        end = value.get('end', 0.0)
        text_list = value.get('text', [])

        # Label Studio stores text as a list ["Text content"]
        if len(text_list) > 1:
          # Should make this a warning in case we do have such cases
          # in our labels by mistake.
          raise ValueError('Only expect one transcription per segment.')

        text_content = text_list[0] if text_list else ""

        # NeMo requires 'duration' and 'offset' for segmented audio
        duration = end - start

        # Skip invalid segments
        if duration <= 0 or not text_content.strip():
          continue

        # Construct the NeMo entry
        manifest_entry = {
          "audio_filepath": audio_filepath,
          "text": text_content,
          "duration": round(duration, 4),
          "offset": round(start, 4),
          "lang": "en",  # transcribe_speech.py reads this field off to configure models
        }

        # Write as a JSON line
        out.write(json.dumps(manifest_entry) + '\n')

  print(f"Successfully converted {input_file} to {output_file}")

In [ ]:
convert_to_nemo_manifest(input_json, output_jsonl, label_studio_data_prefix, actual_data_prefix)

Successfully converted /usr/local/google/home/varungulshan/radio-transcription/model/data/label_studio_exports/playground_export.json to /usr/local/google/home/varungulshan/radio-transcription/model/data/manifests/playground_export_nemo.json
